In [2]:
import torch
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
%load_ext autotime

# ── Config ─────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DEVICE   = "cuda"
DTYPE    = torch.float16   # use float16 if bfloat16 unsupported on your GPU


# ── Load model (do once, reuse for many meshes) ────────────────────────────
def load_model(model_id: str = MODEL_ID):
    print(f"Loading {model_id} ...")
    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",      # auto-assigns layers across GPU (and CPU if needed)
    )
    model.eval()
    print("Model ready.")
    return model, processor




The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 465 μs (started: 2026-04-28 21:48:09 +07:00)


In [3]:
def load_and_resize(img_path, image_size):
    img = Image.open(img_path).convert("RGB")
    img.thumbnail(image_size)
    return img

def run_multimodal_prompt(
    model,
    processor,
    prompt,
    image_paths=None,
    max_new_tokens=256,
    image_size=(256,512)
):
    content = []

    # Add images
    if image_paths:
        for path in image_paths:
            img = load_and_resize(path, image_size)
            content.append({"type": "image", "image": img})

    # Add text
    content.append({"type": "text", "text": prompt})

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    # Build input
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[c["image"] for c in content if c["type"] == "image"] or None,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,  
        )

    # Decode
    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return output_text


time: 486 μs (started: 2026-04-28 21:48:11 +07:00)


In [4]:
model, processor = load_model()

Loading Qwen/Qwen2.5-VL-7B-Instruct ...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model ready.
time: 3min 31s (started: 2026-04-28 21:48:14 +07:00)


In [5]:
out = run_multimodal_prompt(
    model,
    processor,
    "what is physic ?",max_new_tokens=516*2
)

print(out)

Physics is a fundamental science that studies the basic principles governing matter and energy in the universe. It seeks to understand how things move, why objects have mass, and what forces interact with them. Physics encompasses a wide range of topics and can be divided into several branches:

1. Classical Mechanics: This branch deals with the motion of objects under the influence of forces.

2. Thermodynamics: It studies heat, work, and energy transformations, focusing on systems and their properties.

3. Electromagnetism: This field explores the interactions between electrically charged particles and fields, including light.

4. Optics: It examines the behavior of light and its interaction with matter.

5. Quantum Mechanics: This branch describes the behavior of particles at the smallest scales, such as atoms and subatomic particles.

6. Relativity: Albert Einstein's theories of special relativity and general relativity explore the effects of velocity and gravity on time and space.

In [7]:
out = run_multimodal_prompt(
    model,
    processor,
    "what is the color of the kitten eyes ?",
    image_paths=["silver-tabby-cat-sitting-on-green-background-free-photo.jpg"],max_new_tokens=516,image_size =(512,512)
)

print(out)

The kitten in the picture has yellow eyes.
time: 533 ms (started: 2026-04-28 21:54:17 +07:00)


In [9]:
out = run_multimodal_prompt(
    model,
    processor,
    """These images show multiple views of the same fractured 3D object. 
The orange/red-highlighted regions indicate the fracture surface — 
the face where the a smaller piece have broke off from the object.

Please describe:
1. What the overall object appears to be (shape, category, likely material)
2. The approximate size and position of the fracture surface relative to 
   the whole object (like 30%)
3. The fracture geometry — is it flat, jagged, curved, or irregular?
4. Which part of the missing piece of this object likely came from 
   (e.g. "a corner piece", "a side slab", "a central chunk")
5. The orientation of the break — which direction did the object split 
   (e.g. "broke horizontally", "diagonal shear", "split along a vertical plane")

Be specific about what you can see across all views combined. combine all of that into 1 line caption""",
    image_paths=["out/everyday/Vase/6af347c8e74b5b715d878ba9ec3c0d6a/fractured_25/piece_0_frac_view_frac02.png","out/everyday/Vase/6af347c8e74b5b715d878ba9ec3c0d6a/fractured_25/piece_0_frac_view_frac04.png"],max_new_tokens=516,image_size =(516,516)
)

print(out)

The 3D object appears to be a stylized flower pot or vase with a wide base tapering towards a narrower neck, made of ceramic or clay. The fracture surface is located around 30% from the top, indicating the object likely broke from a central chunk of the body, with the missing piece resembling a section of the main body rather than a corner or side slab. The fracture geometry is jagged, suggesting either a diagonal shear or split along a vertical plane, though the specific direction cannot be definitively determined without additional context.
time: 2.57 s (started: 2026-04-24 12:55:11 +07:00)


In [7]:
from transformers import Qwen2_5_VLForConditionalGeneration
llm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",      # auto-assigns layers across GPU (and CPU if needed)
    )
print(llm.model.model.embed_tokens) 

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

AttributeError: 'Qwen2_5_VLModel' object has no attribute 'model'

time: 8.55 s (started: 2026-04-27 02:47:07 +07:00)
